In [ ]:
import sys; sys.path.append('..')
import MeshFEM, mesh, visualization, benchmark
import fd_validation, inflation
import pickle, numpy as np
import gzip

origMesh, iwbv, currMesh = pickle.load(gzip.open('data/fusing_smoothness_validation_data.pkl.gz', 'rb'))

In [ ]:
isheet = inflation.InflatableSheet(origMesh, iwbv)
fcs = inflation.FusingCurveSmoothness(isheet)

In [ ]:
class fdValidationWrapper:
    def __init__(self, fcs, currMesh, origMesh):
        self.fcs = fcs
        self.currMesh = currMesh.copy()
        self.origMesh = origMesh
    def energy(self):
        return self.fcs.energy(self.currMesh, self.origMesh)
    def numVars(self): return self.currMesh.numVertices() * 2
    def getVars(self):
        return self.currMesh.vertices()[:, 0:2].ravel()
    def setVars(self, v):
        return self.currMesh.setVertices(np.pad(v.reshape((-1, 2)), [(0, 0), (0, 1)]))
    def gradient(self):
        return self.fcs.gradient(self.currMesh, self.origMesh).ravel()

In [ ]:
fdw = fdValidationWrapper(fcs, currMesh, origMesh)

In [ ]:
fcs.dirichletWeight = 1.0
fcs.laplacianWeight = 1.0
fcs.lengthScaleSmoothingWeight = 1.0
fcs.curvatureWeight = 1.0

In [ ]:
fcs.dirichletWeight = 0.0
fcs.laplacianWeight = 1.0
fcs.lengthScaleSmoothingWeight = 1.0
fcs.curvatureWeight = 1.0

In [ ]:
benchmark.reset()
fd_validation.gradConvergencePlot(fdw)
benchmark.report()

In [ ]:
fdw.energy()

In [ ]:
fcs.curvatureSmoothingActivationThreshold = 1e-1

In [ ]:
fdw.energy()

In [ ]:
V = origMesh.vertices()
polylines = fcs.boundaryLoops + fcs.wallCurves
visualization.plot_polylines([V[p] for p in polylines], width=16, height=9)

In [ ]:
#visualization.plot_2d_mesh(origMesh, pointList=[4115, 12246, 7386, 8184, 17021, 19049, 23193, 36063, 25990, 35796], width=16, height=8, triEdgeWidth=0.1, bbox=[[-10,0], [-5,5]])